In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import os
import datetime
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from time import sleep

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'RS NBSRS'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':','.')[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running RS NBSRS Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')
base_url = 'https://webappcenter.nbs.rs'

regdict={
    regulatorName + ' 1': 'https://webappcenter.nbs.rs/WebApp/FinInstBank/LegalEntityFinInstitution/Index?',
    regulatorName + ' 2': 'https://webappcenter.nbs.rs/WebApp/FinInstLeasing/LegalEntityFinInstitution/Index?',
    regulatorName + ' 3': 'https://webappcenter.nbs.rs/WebApp/InsuranceMarket/Home/Index?isSearchExecuted=false',
    regulatorName + ' 4': 'https://webappcenter.nbs.rs/WebApp/FinInstValPanFund/LegalEntityFinInstitution/Index?',
    regulatorName + ' 5': 'https://webappcenter.nbs.rs/WebApp/FinInstElectronicMoney/LegalEntityFinInstitution/Index?',
    regulatorName + ' 6': 'https://webappcenter.nbs.rs/WebApp/FinInstPayment/LegalEntityFinInstitution/Index?',
    regulatorName + ' 7': 'https://webappcenter.nbs.rs/WebApp/FinInstElectronicMoneyForeing/LegalEntityFinInstitution/Index?',
}

Typology={
    regulatorName + ' 1': 'List of Banks',
    regulatorName + ' 2': 'List of Lessors',
    regulatorName + ' 3': 'Insurance Market Participants',
    regulatorName + ' 4': 'List of VPF Management Companies',
    regulatorName + ' 5': 'Register of electronic money institutions',
    regulatorName + ' 6': 'Register of payment institutions',
    regulatorName + ' 7': 'List of electronic money institutions from third countries',
}

insurance_category_urls = {
    'Insurance and reinsurance companies': 'https://webappcenter.nbs.rs/WebApp/InsuranceMarket/InsuranceMarketEntity/IndexNoSearchPanel?TypeID=10&LegalEntityTypeID=PravnoLice&IsCombined=true',
    'Insurance brokerage companies': 'https://webappcenter.nbs.rs/WebApp/InsuranceMarket/InsuranceMarketEntity/IndexNoSearchPanel?TypeID=30&LegalEntityTypeID=PravnoLice&IsCombined=true',
    'Insurance agency companies': 'https://webappcenter.nbs.rs/WebApp/InsuranceMarket/InsuranceMarketEntity/IndexNoSearchPanel?TypeID=40&LegalEntityTypeID=PravnoLice&IsCombined=true',
}


In [ ]:
#------------------------------------------------ Begin_Function ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key] = sqldict[key] + empty
    return sqldict


def clean(value):
    if value is None:
        return ''
    return ' '.join(str(value).replace('\r', ' ').replace('\n', ' ').split())


def get_soup(url, max_attempts=4):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
        'Accept-Language': 'sr-RS,sr;q=0.9,en;q=0.8',
        'Connection': 'close',
    }
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(url, headers=headers, timeout=60)
            response.raise_for_status()
            response.encoding = response.apparent_encoding
            return BeautifulSoup(response.text, 'html.parser')
        except requests.exceptions.RequestException as error:
            last_error = error
            if attempt == max_attempts:
                break
            wait_seconds = attempt * 2
            print(f"[WARN] : Request failed for {url}; retrying in {wait_seconds}s ({attempt}/{max_attempts})")
            sleep(wait_seconds)
    raise last_error


def get_field_value(container):
    field = container.find(['input', 'textarea', 'select'])
    if field:
        if field.name == 'input':
            return clean(field.get('value', ''))
        if field.name == 'textarea':
            return clean(field.get_text(' ', strip=True))
        selected = field.find('option', selected=True)
        return clean(selected.get_text(' ', strip=True) if selected else field.get_text(' ', strip=True))

    link = container.find('a', href=True)
    if link:
        href = clean(link.get('href'))
        text = clean(link.get_text(' ', strip=True))
        if href.startswith('mailto:'):
            return href.replace('mailto:', '')
        if href.startswith('http'):
            return href
        return text

    value_holder = container.find(class_='form-control')
    if value_holder:
        return clean(value_holder.get_text(' ', strip=True))
    return ''


def get_label_values(soup):
    details = {}
    for label in soup.find_all('label'):
        label_name = clean(label.get_text(' ', strip=True))
        container = label.find_parent('div')
        if not label_name or not container:
            continue
        details[label_name] = get_field_value(container)
    return details


def extract_table_records(url, typology=None):
    soup = get_soup(url)
    table = soup.find('table')
    if not table:
        return []

    rows = table.find_all('tr')
    if not rows:
        return []

    headers = [clean(cell.get_text(' ', strip=True)) for cell in rows[0].find_all(['th', 'td'])]
    records = []

    for row in rows[1:]:
        cells = row.find_all('td')
        if not cells:
            continue

        row_values = {}
        for index, cell in enumerate(cells):
            if index < len(headers) and headers[index]:
                row_values[headers[index]] = clean(cell.get_text(' ', strip=True))

        detail_url = ''
        for link in row.find_all('a', href=True):
            href = link.get('href', '')
            link_text = clean(link.get_text(' ', strip=True))
            if 'Details' in href or 'Детаљи' in link_text:
                detail_url = urljoin(base_url, href)
                break

        detail_values = {}
        if detail_url:
            detail_values = get_label_values(get_soup(detail_url))
            sleep(0.2)

        records.append({
            'row': row_values,
            'details': detail_values,
            'detail_url': detail_url or url,
            'typology': typology,
        })

    return records


def first_non_empty(*values):
    for value in values:
        value = clean(value)
        if value:
            return value
    return ''


def get_internal_id(details, row):
    for label, id_type in [
        ('Матични број', 'National ID'),
        ('Регистарски број', 'Registration number'),
        ('Регистрациони број у националном регистру', 'National register number'),
    ]:
        value = first_non_empty(details.get(label), row.get(label))
        if value:
            return value, id_type
    return '', ''


def append_sql_record(record, reg, list_name, list_code):
    row = record['row']
    details = record['details']
    name = first_non_empty(
        details.get('Назив'),
        details.get('Пословно име'),
        row.get('Назив'),
        row.get('Пословно име'),
        row.get('Назив правног лица'),
    )
    if not name:
        return False

    internal_id_1, internal_id_1_type = get_internal_id(details, row)
    internal_id_2 = first_non_empty(details.get('Порески број'), details.get('ПИБ'))
    internal_id_3 = first_non_empty(details.get('SWIFT'))

    sqldict['Name'].append(name)
    sqldict['InternalID_1'].append(internal_id_1)
    sqldict['InternalID_1_type'].append(internal_id_1_type)
    sqldict['InternalID_2'].append(internal_id_2)
    sqldict['InternalID_2_type'].append('Tax number' if internal_id_2 else '')
    sqldict['BIC SWIFT Code'].append(internal_id_3)
    
    # sqldict['InternalID_3_type'].append('SWIFT' if internal_id_3 else '')
    # sqldict['License_Type'].append(first_non_empty(
    #     details.get('Број решења дозволе за рад'),
    #     details.get('Број дозволе за рад'),
    #     details.get('Врста регистра'),
    #     row.get('Врста регистра'),
    #     row.get('Надлежни национални регулатор'),
    # ))
    sqldict['Address_1'].append(first_non_empty(details.get('Адреса'), details.get('Адреса седишта'), row.get('Адреса'), row.get('Седиште')))
    sqldict['City'].append(first_non_empty(details.get('Град'), details.get('Седиште')))
    sqldict['Zip'].append(first_non_empty(details.get('Поштански број')))
    sqldict['Cntry'].append('RS' if list_code != '7' else '')
    sqldict['Phone'].append(first_non_empty(details.get('Телефон'), row.get('Телефон')))
    sqldict['Fax'].append(first_non_empty(details.get('Телефакс'), row.get('Телефакс')))
    sqldict['Website'].append(first_non_empty(details.get('Веб сајт'), details.get('WWW')))
    sqldict['Email'].append(first_non_empty(details.get('И-мејл'), details.get('Имејл')))
    sqldict['Typology'].append(first_non_empty(record.get('typology'), row.get('Врста регистра'), list_name))
    sqldict['RegulationType'].append('Regulated')
    sqldict['RegulationDate'].append(first_non_empty(
        details.get('Датум решења дозволе за рад'),
        details.get('Датум издавања дозволе'),
        row.get('Датум пријема обавештења из члана 225. ЗПУ'),
    ))
    sqldict['RegCtry'].append('RS')
    sqldict['RegCode'].append('NBSRS')
    sqldict['ListCode'].append(list_code)
    # sqldict['ListLanguage'].append('Serbian')
    sqldict['ListName'].append(list_name)
    sqldict['ListProcessDate'].append(processdate)

    bourange_same_length_array(sqldict)
    return True

In [5]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, (reg, url) in enumerate(regdict.items(), start=1):
    list_name = Typology[reg]
    list_code = reg.split(' ')[-1]
    print(f"[INFO] : Working {k}/{len(regdict)} _({reg})_ - {list_name}")

    if list_code == '3':
        records = []
        for category_name, category_url in insurance_category_urls.items():
            print(f"[INFO] : Working insurance category - {category_name}")
            category_records = extract_table_records(category_url, category_name)
            print(f"[INFO] : Found {len(category_records)} entities in {category_name}")
            records.extend(category_records)
    else:
        records = extract_table_records(url, list_name)
        print(f"[INFO] : Found {len(records)} entities on {reg}")

    added_count = 0
    for record in records:
        if append_sql_record(record, reg, list_name, list_code):
            added_count += 1

    print(f"[INFO] : Added {added_count} SQL rows for {reg}")
    sleep(1)

print(f"[INFO] : Total SQL rows collected: {len(sqldict['Name'])}")

[INFO] : Working 1/7 _(RS NBSRS 1)_ - List of Banks
[INFO] : Found 19 entities on RS NBSRS 1
[INFO] : Added 19 SQL rows for RS NBSRS 1
[INFO] : Working 2/7 _(RS NBSRS 2)_ - List of Lessors
[INFO] : Found 13 entities on RS NBSRS 2
[INFO] : Added 13 SQL rows for RS NBSRS 2
[INFO] : Working 3/7 _(RS NBSRS 3)_ - Insurance Market Participants
[INFO] : Working insurance category - Insurance and reinsurance companies
[INFO] : Found 20 entities in Insurance and reinsurance companies
[INFO] : Working insurance category - Insurance brokerage companies
[INFO] : Found 50 entities in Insurance brokerage companies
[INFO] : Working insurance category - Insurance agency companies
[INFO] : Found 30 entities in Insurance agency companies
[INFO] : Added 100 SQL rows for RS NBSRS 3
[INFO] : Working 4/7 _(RS NBSRS 4)_ - List of VPF Management Companies
[INFO] : Found 4 entities on RS NBSRS 4
[INFO] : Added 4 SQL rows for RS NBSRS 4
[INFO] : Working 5/7 _(RS NBSRS 5)_ - Register of electronic money institut

In [10]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

name_cleanup_by_typology = {
    'Insurance and reinsurance companies': ['Akcionarsko društvo za osiguranje'],
    'Insurance brokerage companies': [
        'Društvo za posredovanje u oiguranju',
        'Društvo za posredovanje u osiguranju',
    ],
    'Insurance agency companies': ['Društvo za zastupanje u osiguranju'],
}

for typology, phrases in name_cleanup_by_typology.items():
    mask = df['Typology'].eq(typology)
    for phrase in phrases:
        df.loc[mask, 'Name'] = df.loc[mask, 'Name'].str.replace(phrase, '', regex=False).str.strip()

quoted_name_typologies = [
    'Insurance brokerage companies',
    'Insurance agency companies',
]
quoted_name_mask = df['Typology'].isin(quoted_name_typologies)
quoted_names = df.loc[quoted_name_mask, 'Name'].str.extract(r'"([^"]+)"', expand=False)
df.loc[quoted_name_mask & quoted_names.notna(), 'Name'] = quoted_names[quoted_names.notna()].str.strip()

for contact_column in ['Phone', 'Fax']:
    contact_mask = df[contact_column].fillna('').astype(str).str.strip().str.len().gt(3)
    df.loc[~contact_mask, contact_column] = ''

df.to_excel(filename, 'SQL Ready', index=False)
sleep(3)
print(f"[INFO] : Excel file '{filename}' saved successfully")

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_29508\3147178665.py:31: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


[INFO] : Excel file 'RS NBSRS SQL Ready 2026-04-27 21.45.08.xlsx' saved successfully


In [7]:
#------------------------------------------------ Optional data quality checks ----------------------------------------
df.head()

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,List of Banks,,Addiko Bank a.d. Beograd,7726716,National ID,100228215,Tax number,...,,HAABRSBG,,,,,,,,
1,,,,List of Banks,,AikBank akcionarsko društvo Beograd,6876366,National ID,100618836,Tax number,...,,AIKBRS22,,,,,,,,
2,,,,List of Banks,,ALTA banka a.d. Beograd,7074433,National ID,100001829,Tax number,...,,JMBNRSBG,,,,,,,,
3,,,,List of Banks,,Adriatic Bank akcionarsko društvo Beograd,7534183,National ID,100003148,Tax number,...,,LIKIRSBG,,,,,,,,
4,,,,List of Banks,,API Bank akcionarsko društvo Beograd,20439866,National ID,105701111,Tax number,...,,APIBRSBG,,,,,,,,
